# Lab 03 · Classification & Rigorous Evaluation
**Advanced Data Mining — M.Sc. Data Science** · Dr. Ali Louati · Prince Sattam Bin Abdulaziz University
Companion to **Chapter 3**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alilouati/advanced-data-mining/blob/main/labs/Lab03_Classification_and_Evaluation.ipynb)

**In this lab you will:**
- train k-NN, Naïve Bayes, logistic regression and SVM classifiers
- compare them with stratified cross-validation
- read confusion matrices, ROC and precision–recall curves
- visualise decision boundaries and tune k

> **How to use:** run the cells from top to bottom (`Shift + Enter`). Cells marked ✏️ are for you to modify.
> Every step works with the built-in data set; set `USE_MY_CSV = True` to try your own CSV file.

In [ ]:
# Setup: libraries used in this lab (all pre-installed on Google Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (8, 4.5)
np.random.seed(42)

## 1 · Data: breast-cancer diagnosis
569 tumours, 30 numeric features, target = malignant (0) or benign (1).

In [ ]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
df = data.frame
TARGET = "target"
print(df.shape); df[TARGET].value_counts()

In [ ]:
# ✏️ Optional: use your own data set instead of the built-in one
USE_MY_CSV = False   # change to True, run the cell, then choose a .csv file

if USE_MY_CSV:
    from google.colab import files          # works on Google Colab
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))
    print("Loaded your file:", df.shape)
else:
    print("Using the built-in data set (breast cancer).")

In [ ]:
# ✏️ If you loaded your own CSV, set TARGET to the name of your label column
X = df.drop(columns=[TARGET]).select_dtypes("number")
y = df[TARGET]
sns.countplot(x=y); plt.title("Class distribution"); plt.show()

## 2 · Train / test split (stratified)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
print("train:", X_train.shape, " test:", X_test.shape)

## 3 · Four classifiers inside pipelines
Scaling is part of each pipeline, so it is fitted on training data only (no leakage).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    "k-NN (k=5)":   make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes":  make_pipeline(StandardScaler(), GaussianNB()),
    "Logistic reg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "SVM (RBF)":    make_pipeline(StandardScaler(), SVC(probability=True)),
}

## 4 · Compare with 10-fold stratified cross-validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

rows = []
for name, model in models.items():
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=["accuracy", "f1", "roc_auc"])
    rows.append({"model": name,
                 "accuracy": res["test_accuracy"].mean(),
                 "F1": res["test_f1"].mean(),
                 "ROC-AUC": res["test_roc_auc"].mean(),
                 "F1 std": res["test_f1"].std()})
results = pd.DataFrame(rows).set_index("model").round(3)
display(results)
results[["accuracy", "F1", "ROC-AUC"]].plot(kind="bar", ylim=(0.85, 1.0), rot=0)
plt.title("10-fold CV scores"); plt.show()

## 5 · Final evaluation on the test set

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

best = models["Logistic reg"]          # ✏️ try another model
best.fit(X_train, y_train)
y_pred = best.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Confusion matrix"); plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, model in models.items():
    model.fit(X_train, y_train)
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], "k--"); axes[0].set_title("ROC curves")
axes[1].set_title("Precision-Recall curves"); plt.show()

## 6 · Visualising decision boundaries (2 features)

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay
two = X.columns[:2]                     # ✏️ pick two other features
X2 = X_train[two]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X2, y_train)
    DecisionBoundaryDisplay.from_estimator(model, X2, ax=ax, alpha=0.4, cmap="coolwarm", response_method="predict")
    ax.scatter(X2.iloc[:, 0], X2.iloc[:, 1], c=y_train, cmap="coolwarm", s=10, edgecolor="k", linewidth=0.2)
    ax.set_title(name); ax.set_xlabel(two[0]); ax.set_ylabel(two[1])
plt.tight_layout(); plt.show()

## 7 · Choosing k for k-NN (validation curve)

In [ ]:
from sklearn.model_selection import cross_val_score
ks = range(1, 31, 2)
scores = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)),
                          X_train, y_train, cv=cv).mean() for k in ks]
plt.plot(list(ks), scores, marker="o")
plt.xlabel("k"); plt.ylabel("CV accuracy"); plt.title("k-NN: accuracy vs. k"); plt.show()
print("best k =", list(ks)[int(np.argmax(scores))])

## ✏️ Exercises
1. Which model has the best F1? Is the difference larger than the standard deviation?
2. Remove `StandardScaler` from the k-NN pipeline. What happens and why?
3. Change the decision threshold of logistic regression to 0.3 (`predict_proba`) and recompute recall.
4. Run the notebook on your own labelled CSV.